In [112]:
import openai
import json
import requests

client = openai.OpenAI()

fetch_api = "https://nomad-movies.nomadcoders.workers.dev"
messages = []

In [113]:
def get_similar_movies(id: str):
    return requests.get(f"{fetch_api}/movies/{id}/similar").text

In [114]:
def get_popular_movies():
    
    movie_list = []
    response = requests.get(fetch_api+"/movies")
    results = json.loads(response.text)

    for movie in results:
        movie_list.append({
            "id": movie.get("id"),
            "name": movie.get("title"),
            "genre_ids": movie.get("genre_ids"),
            "popularity": movie.get("popularity"),
            "overview": movie.get("overview")
        })
    return json.dumps(movie_list)

In [115]:
def get_movie_details(id: str):
    return requests.get(f"{fetch_api}/movies/{id}").text

In [116]:
FUNCTION_MAP = {
    'get_popular_movies': get_popular_movies,
    'get_movie_details':get_movie_details,
    'get_similar_movies':get_similar_movies
}

In [117]:
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "get_popular_movies",
            "description": "Fetches popular movies.",
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_movie_details",
            "description": "Fetches detailed movie information.",
            "parameters": {
                "type": "object",
                "properties": {
                    "id": {
                        "type": "string",
                        "description": "The id of the movie to get detail.",
                    }
                },
                "required": ["id"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_similar_movies",
            "description": "Fetches cast and crew information.",
            "parameters": {
                "type": "object",
                "properties": {
                    "id": {
                        "type": "string",
                        "description": "The id the movie to get cast and crew information.",
                    }
                },
                "required": ["id"],
            },
        },
    },
]

In [118]:
from openai.types.chat import ChatCompletionMessage

def process_ai_response(message: ChatCompletionMessage):

    if message.tool_calls:
        messages.append(
            {
                "role": "assistant",
                "content": message.content or "",
                "tool_calls":[
                    {
                        "id": tc.id,
                        "type": "function",
                        "function":{
                            "name": tc.function.name,
                            "arguments": tc.function.arguments,
                        }
                    } for tc in message.tool_calls
                ]
            } 
        )

        for tc in message.tool_calls:
            function_name = tc.function.name
            arguments = tc.function.arguments
            parsed_args = json.loads(arguments) if arguments else {}
            args_str = ", ".join(str(v) for v in parsed_args.values())

            print (f"AI: [{function_name}({args_str})] 호출")

            try:
                arguments = json.loads(arguments)
            except json.JSONDecodeError:
                arguments = {}
            
            function_to_run = FUNCTION_MAP.get(function_name)

            result = function_to_run(**arguments)

            messages.append({
                "role": "tool",
                "tool_call_id": tc.id,
                "name": tc.function.name,
                "content": result,
            })
        call_ai()
    else:
        messages.append({
            "role": "assistant",
            "content": message.content
        })
        print (f"AI: {message.content}")

def call_ai():
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
        tools=TOOLS
    )
    process_ai_response(response.choices[0].message)

In [119]:
while True:
    message = input("Send me a message")

    if message.lower() in ["quit","q"]:
        break
    else:
        messages.append({"role":"user","content":message})
        print (f"User: {message}")
        call_ai()
        

User: 영화 추천해줘
AI: [get_popular_movies()] 호출
AI: 다음은 추천할 만한 인기 영화들입니다:

1. **Shelter**
   - **장르**: 액션, 범죄, 스릴러
   - **개요**: 외딴 섬에서 자발적으로 은신 중인 한 남자가 폭풍에서 젊은 소녀를 구하며 그의 과거와 연관된 적들로부터 그녀를 보호하기 위해 세상으로 나아가게 되는 이야기.

2. **Mercy**
   - **장르**: 공상과학, 액션, 스릴러
   - **개요**: 가까운 미래, 한 형사가 아내를 살해한 혐의로 재판에 서고, 90분 안에 고급 AI 판사에게 자신의 무죄를 입증해야 하는 긴장감 폭발적인 이야기.

3. **The Bluff**
   - **장르**: 액션, 스릴러
   - **개요**: 평화로운 섬에서의 일상이 복수심에 가득 찬 전 선장에 의해 깨지면서, 전문적인 해적 출신의 여성 주인공이 가족을 구하기 위해 자신의 과거와 대면해야 하는 이야기.

4. **Scream 7**
   - **장르**: 공포, 미스터리, 범죄
   - **개요**: 새로운 고스트페이스 킬러가 나타나면서, 시드니가 자신의 가족을 보호하기 위해 과거의 두려움과 맞서 싸워야 하는 스릴 넘치는 이야기.

5. **Avatar: Fire and Ash**
   - **장르**: 공상과학, 모험, 판타지
   - **개요**: 전쟁 이후, 제이크 쏠리와 네이티리가 새로운 위협에 맞서기 위해 싸우는 이야기를 다룬다.

이 영화들 중에 관심이 가는 것이 있나요? 더 자세한 정보를 원하시면 말씀해 주세요!
User: Mercy에 대해서 더 알려줘.
AI: [get_movie_details(1236153)] 호출
AI: 영화 **Mercy**에 대한 자세한 정보는 다음과 같습니다:

- **제목**: Mercy
- **개봉일**: 2026년 1월 20일
- **장르**: 공상과학, 액션, 스릴러
- **러닝타임**: 100분
- **감독**: 정보 없음
- **제작사**: 
  